Load data and import

In [ ]:
import pandas as pd
from IPython.display import display, Markdown
from pathlib import Path
from IPython.display import display, Markdown

In [ ]:


df = pd.read_parquet("recipes_cleaned.parquet")
hybrid_pairs = pd.read_parquet("hybrid_duplicate_pairs.parquet")

LABEL_PATH = Path("duplicate_likert_labels.csv")

if LABEL_PATH.exists():
    labels = pd.read_csv(LABEL_PATH)
else:
    labels = pd.DataFrame(columns=[
        "recipe_id",
        "match_id",
        "semantic_score",
        "ingredient_jaccard",
        "title_similarity",
        "hybrid_score",
        "likert_label",
        "notes"
    ])

labels.head()

build unlabelled queue

In [ ]:
already_labeled = set(
    zip(labels["recipe_id"].astype(int), labels["match_id"].astype(int))
)

unlabeled = hybrid_pairs[
    ~hybrid_pairs.apply(
        lambda row: (int(row["recipe_id"]), int(row["match_id"])) in already_labeled,
        axis=1
    )
].copy()

unlabeled = unlabeled.sample(frac=1, random_state=42).reset_index(drop=True)

print("Already labeled:", len(labels))
print("Remaining unlabeled:", len(unlabeled))

Labelling

In [ ]:
if len(unlabeled) == 0:
    print("All examples labeled.")
else:
    row = unlabeled.iloc[0]

    recipe_id = int(row["recipe_id"])
    match_id = int(row["match_id"])

    display(Markdown("# Recipe Pair"))

    print("=" * 100)
    print("SIMILARITY FEATURES")
    print(f"Semantic Score:      {row['semantic_score']:.4f}")
    print(f"Ingredient Jaccard:  {row['ingredient_jaccard']:.4f}")
    print(f"Title Similarity:    {row['title_similarity']:.4f}")
    print(f"Hybrid Score:        {row['hybrid_score']:.4f}")

    print("\n" + "=" * 100)
    print("RECIPE A")
    print("Title:", df.iloc[recipe_id]["title"])
    print("\nIngredients:")
    print(df.iloc[recipe_id]["ingredients_clean"])
    print("\nDirections:")
    print(df.iloc[recipe_id]["directions_clean"])

    print("\n" + "=" * 100)
    print("RECIPE B")
    print("Title:", df.iloc[match_id]["title"])
    print("\nIngredients:")
    print(df.iloc[match_id]["ingredients_clean"])
    print("\nDirections:")
    print(df.iloc[match_id]["directions_clean"])

    print("\n" + "=" * 100)
    print("LIKERT SCALE")
    print("1 = completely different")
    print("2 = conceptually related")
    print("3 = variant")
    print("4 = near duplicate")
    print("5 = essentially identical")

    label = int(input("\nEnter label (1-5): "))
    notes = input("Optional notes: ")

    new_row = pd.DataFrame([{
        "recipe_id": recipe_id,
        "match_id": match_id,
        "semantic_score": row["semantic_score"],
        "ingredient_jaccard": row["ingredient_jaccard"],
        "title_similarity": row["title_similarity"],
        "hybrid_score": row["hybrid_score"],
        "likert_label": label,
        "notes": notes
    }])

    labels = pd.concat([labels, new_row], ignore_index=True)
    labels.to_csv(LABEL_PATH, index=False)

    unlabeled = unlabeled.iloc[1:].reset_index(drop=True)

    print("\nSaved.")
    print("Total labels:", len(labels))
    print("Remaining unlabeled:", len(unlabeled))